# Tracing Fine-Tuned Models with Opik and the CometML Model Registry

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/comet-ml/opik-examples/blob/main/guides/tracing_finetuned_models/tracing_finetuned_models.ipynb)

This notebook shows the full workflow from training to traced inference:

1. Fine-tune `distilgpt2` on an instruction dataset using HuggingFace trl
2. Register the trained model to the CometML Model Registry
3. Fetch the registered model by name and version
4. Run inference inside an Opik-traced function that links every prediction back to the exact model version that produced it

```
Section 1                       Section 2
train_and_register              fetch_and_trace
       │                               │
       │  fine-tunes distilgpt2        │  fetches registered model
       │  logs checkpoints             │  runs Opik-traced inference
       ▼                               ▼
 CometML Model Registry ──────► Opik trace
   sft-distilgpt2 v1.0.0          metadata.model_registry_url
```

**Already have a registered model?** Skip straight to [Section 2](#section-2).

---

**Recommended runtime:** GPU (T4 or better). In Colab: *Runtime → Change runtime type → T4 GPU*.

## 0. Setup

In [ ]:
%pip install comet_ml transformers trl datasets torch opik --quiet

> **Note:** If you just installed the packages for the first time, restart the Colab runtime now (*Runtime → Restart session*) then run all cells again from the top.

In [ ]:
import os
import getpass

#TODO: Add Opik and COMET api key and workspace to the the same

# Locally: set them as environment variables before running the script, e.g.:
# export COMET_API_KEY=your_comet_api_key
# export COMET_WORKSPACE=your_comet_workspace
COMET_API_KEY   = os.environ.get('COMET_API_KEY', '') or getpass.getpass('Enter your CometML API key: ')
COMET_WORKSPACE = os.environ.get('COMET_WORKSPACE', '') or getpass.getpass('Enter your CometML workspace: ')
OPIK_API_KEY    = os.environ.get('OPIK_API_KEY', '') or getpass.getpass('Enter your Opik API key: ')
OPIK_WORKSPACE  = os.environ.get('OPIK_WORKSPACE', '') or getpass.getpass('Enter your Opik workspace: ')

# Update these to match your use case.
REGISTRY_NAME = 'sft-distilgpt2'
MODEL_VERSION = '1.0.0'
OPIK_PROJECT  = 'tracing-finetuned-models'

print(f'CometML workspace : {COMET_WORKSPACE or "(not set)"}')
print(f'Opik workspace    : {OPIK_WORKSPACE or "(not set)"}')

CometML workspace : leo-breedt-5463
Opik workspace    : leo-breedt-5463


---
## 1. Train and register

> **Skip this section** if you already have a model registered in the CometML Model Registry. Jump to [Section 2](#section-2) and set `REGISTRY_NAME` and `MODEL_VERSION` to match your existing model.

This section fine-tunes `distilgpt2` on the [OpenAssistant Guanaco](https://huggingface.co/datasets/timdettmers/openassistant-guanaco) instruction dataset using supervised fine-tuning (SFT) via HuggingFace trl. A checkpoint is logged to a CometML experiment after each epoch; the final model is registered to the CometML Model Registry.

**Requires:** `COMET_API_KEY` and `COMET_WORKSPACE` set above.

> **Important:** `comet_ml` must be imported before `transformers` and `trl`. The CometML Trainer integration hooks in at import time. Run the cell below before any other imports.

In [3]:
# comet_ml MUST be imported first.
os.environ['COMET_API_KEY'] = COMET_API_KEY or ''
import comet_ml  # noqa: F401 — triggers auto-integration with Trainer

In [4]:
import shutil
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainerCallback
from trl import SFTTrainer, SFTConfig

MODEL_NAME      = 'distilgpt2'
NUM_EPOCHS      = 3
BATCH_SIZE      = 4
LEARNING_RATE   = 2e-5
MAX_SEQ_LENGTH  = 256
OUTPUT_DIR      = './results_sft'
FINAL_MODEL_DIR = './final_model'

/Users/leobreedt/Comet/Repositories/opik-examples/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
dataset       = load_dataset('timdettmers/openassistant-guanaco')
train_dataset = dataset['train'].select(range(400))
eval_dataset  = dataset['test'].select(range(100))

print(f'Train: {len(train_dataset)} examples  |  Eval: {len(eval_dataset)} examples')

Repo card metadata block was not found. Setting CardData to empty.
Generating test split: 100%|██████████| 518/518 [00:00<00:00, 220349.85 examples/s]

Train: 400 examples  |  Eval: 100 examples


In [6]:
tokenizer           = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

train_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

print(f'Model loaded: {MODEL_NAME}')
print(f'Parameters  : {sum(p.numel() for p in train_model.parameters()):,}')

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 25640.85it/s]


Model loaded: distilgpt2
Parameters  : 81,912,576


In [7]:
class CheckpointCallback(TrainerCallback):
    """Logs a model checkpoint to the CometML experiment after each epoch."""

    def on_train_begin(self, args, state, control, **kwargs):
        experiment = comet_ml.get_running_experiment()
        if experiment:
            experiment.add_tag('SFT')

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = int(state.epoch)
        experiment = comet_ml.get_running_experiment()
        if experiment is None or model is None:
            return

        checkpoint_path = f'{args.output_dir}/checkpoint-epoch-{epoch}'
        model.save_pretrained(checkpoint_path)
        tokenizer.save_pretrained(checkpoint_path)

        experiment.log_model(
            name=f'checkpoint_epoch_{epoch}',
            file_or_folder=checkpoint_path,
            metadata={'epoch': epoch},
        )
        shutil.rmtree(checkpoint_path)
        print(f'  → Logged checkpoint_epoch_{epoch} to CometML')

In [8]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to=['comet_ml'],
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field='text',
    packing=False,
)

trainer = SFTTrainer(
    model=train_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    callbacks=[CheckpointCallback()],
)

print('Starting training...')
trainer.train()
print('Training complete.')

/var/folders/4t/8k73_twn4f5b2qhxq_wzq4dc0000gn/T/ipykernel_7309/1905026704.py:1: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(
Tokenizing eval dataset: 100%|██████████| 100/100 [00:00<00:00, 2097.90 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
COMET WARNING: 

Starting training...


COMET INFO: Experiment is live on comet.com https://www.comet.com/leo-breedt-5463/general/d10c5033c1f64bf5abe9fed146eba98b

/Users/leobreedt/Comet/Repositories/opik-examples/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,3.491239,3.765815,3.616502,0.334096,91500.000000
2,3.618924,3.693157,3.477753,0.343718,183000.000000
3,3.683178,3.673179,3.480193,0.344650,274500.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.77it/s]


...

  → Logged checkpoint_epoch_2 to CometML


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.02it/s]
/Users/leobreedt/Comet/Repositories/opik-examples/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.96it/s]


  → Logged checkpoint_epoch_3 to CometML


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.88it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Training complete.


In [9]:
experiment = comet_ml.get_running_experiment()

if experiment:
    train_model.save_pretrained(FINAL_MODEL_DIR)
    tokenizer.save_pretrained(FINAL_MODEL_DIR)

    experiment.log_model(
        name=REGISTRY_NAME,
        file_or_folder=FINAL_MODEL_DIR,
        metadata={'epochs': NUM_EPOCHS, 'base_model': MODEL_NAME},
    )

    experiment.register_model(
        model_name=REGISTRY_NAME,
        registry_name=REGISTRY_NAME,
        version=MODEL_VERSION,
        tags=['sft', MODEL_NAME],
        comment=f'SFT on OpenAssistant Guanaco, {NUM_EPOCHS} epochs',
        public=False,
    )

    comet_workspace = experiment.workspace
    experiment.end()
    shutil.rmtree(FINAL_MODEL_DIR, ignore_errors=True)

    registry_url = f'https://www.comet.com/{comet_workspace}/model-registry/{REGISTRY_NAME}/{MODEL_VERSION}'
    print(f'Model registered: {registry_url}')
    print(f'\nUse in Section 2:')
    print(f'  REGISTRY_NAME = "{REGISTRY_NAME}"')
    print(f'  MODEL_VERSION = "{MODEL_VERSION}"')
else:
    print('No active CometML experiment — check that COMET_API_KEY is set.')

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.84it/s]
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : dizzy_moose_7828
COMET INFO:     url                   : https://www.comet.com/leo-breedt-5463/general/d10c5033c1f64bf5abe9fed146eba98b
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     eval/entropy [3]               : (3.4777532958984376, 3.6165017795562746)
COMET INFO:     eval/loss [3]                  : (3.6731786727905273, 3.765815496444702)
COMET INFO:     eval/mean_token_accuracy [3]   : (0.33409633159637453, 0.3446498554944992)
COMET INFO:     eval/num_tokens [3]            : (91500.0, 274500.0)
COMET INFO:     eval/runtime [3]               : (3.4792, 4.4491)
COMET I

Model registered: https://www.comet.com/None/model-registry/sft-distilgpt2/1.0.0

Use in Section 2:
  REGISTRY_NAME = "sft-distilgpt2"
  MODEL_VERSION = "1.0.0"


---
<a id='section-2'></a>
## 2. Fetch the registered model

Download the registered model from the CometML Model Registry. If you skipped Section 1 and don't have a registered model yet, the cell below falls back to loading `distilgpt2` directly from HuggingFace so you can still follow the Opik tracing steps.

**Requires:** `COMET_API_KEY`, `COMET_WORKSPACE`, `REGISTRY_NAME`, and `MODEL_VERSION` set in the Setup section.

In [ ]:
import comet_ml

MODEL_LOCAL_DIR = './downloaded_model'

if COMET_API_KEY and COMET_WORKSPACE:
    api = comet_ml.API(api_key=COMET_API_KEY)
    print(f'Downloading {REGISTRY_NAME} v{MODEL_VERSION} from CometML Model Registry...')
    registered_model = api.get_model(workspace=COMET_WORKSPACE, model_name=REGISTRY_NAME)
    registered_model.download(version=MODEL_VERSION, output_folder=MODEL_LOCAL_DIR, expand=True)
    model_source = MODEL_LOCAL_DIR
    print(f'Downloaded to {MODEL_LOCAL_DIR}')
else:
    print('[No credentials] Loading distilgpt2 from HuggingFace as a stand-in.')
    model_source = 'distilgpt2'

In [ ]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# The registry download may nest the model inside a subdirectory
# (matching the folder name used at save_pretrained time).
def _resolve_model_path(path: str) -> str:
    """Return the path that actually contains model config files."""
    if os.path.isfile(os.path.join(path, 'config.json')):
        return path
    subdirs = [os.path.join(path, d) for d in os.listdir(path)
               if os.path.isdir(os.path.join(path, d))]
    for sub in subdirs:
        if os.path.isfile(os.path.join(sub, 'config.json')):
            return sub
    return path

resolved_source = _resolve_model_path(model_source) if model_source != 'distilgpt2' else model_source

inference_tokenizer = AutoTokenizer.from_pretrained(resolved_source)
inference_model     = AutoModelForCausalLM.from_pretrained(resolved_source)
inference_model.eval()

print(f'Model ready: {resolved_source}')

---
## 3. Trace inference with Opik

Every call to `generate()` creates an Opik trace. The trace includes the prompt as input, the generated text as output, and metadata that links directly back to the registered model version in CometML.

**Requires:** `OPIK_API_KEY` and `OPIK_WORKSPACE` set in the Setup section. Without them, `@opik.track` still runs but traces are not sent to Opik.

In [ ]:
import opik

if OPIK_API_KEY and OPIK_WORKSPACE:
    os.environ['OPIK_API_KEY']   = OPIK_API_KEY
    os.environ['OPIK_WORKSPACE'] = OPIK_WORKSPACE
    print(f'Opik configured. Traces will appear in project "{OPIK_PROJECT}".')
else:
    print('[No Opik credentials] Traces will not be sent to Opik.')


def _registry_url() -> str:
    workspace = COMET_WORKSPACE or 'your-workspace'
    return f'https://www.comet.com/{workspace}/model-registry/{REGISTRY_NAME}/{MODEL_VERSION}'

In [ ]:
@opik.track(project_name=OPIK_PROJECT)
def generate(prompt: str, max_new_tokens: int = 100) -> str:
    """
    Run inference and attach the registered model version to the Opik trace.
    The model_registry_url field lets you navigate from any trace directly
    to the CometML experiment and checkpoint that produced it.
    """
    opik.update_current_trace(
        metadata={
            'model_registry_name': REGISTRY_NAME,
            'model_version':       MODEL_VERSION,
            'model_registry_url':  _registry_url(),
        }
    )

    inputs = inference_tokenizer(prompt, return_tensors='pt')
    with torch.no_grad():
        output_ids = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=inference_tokenizer.eos_token_id,
            do_sample=False,
        )
    return inference_tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [ ]:
prompts = [
    'The main advantage of fine-tuning a language model is',
    'Supervised fine-tuning works by',
    'To evaluate a fine-tuned model you should',
]

for prompt in prompts:
    response = generate(prompt)
    print(f'Prompt  : {prompt}')
    print(f'Response: {response}')
    print()

In [ ]:
opik.flush_tracker()

if OPIK_API_KEY and OPIK_WORKSPACE:
    opik_url = f'https://www.comet.com/opik/{OPIK_WORKSPACE}/{OPIK_PROJECT}'
    print(f'Traces sent to Opik: {opik_url}')
    print(f'Each trace includes: model_registry_url = {_registry_url()}')
else:
    print('Set OPIK_API_KEY and OPIK_WORKSPACE in the Setup section to send traces to Opik.')